In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [3]:
df = pd.read_csv("../datasets/processed/merged_data.csv")

numeric_features = ["gdp_per_capita", "unemployment_rate", "population", "urban_pct", "asylum_applications"]
categorical_features = ["country_code"]

X = df[numeric_features + categorical_features]

y = df[["heatwave_days", "precip_days_heavy", "dry_days"]]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features),
    ]
)

In [4]:
model = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", LinearRegression())]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [5]:
print("Predictions:", y_pred)

Predictions: [[ 9.27985887e-02  2.25494255e+00  2.13433457e+02]
 [ 8.74100430e-01  3.52473450e+00  2.62935932e+02]
 [ 7.05317885e-01  3.39557169e+00  2.60750375e+02]
 [-7.35867749e-02  2.70082818e+00  2.33908964e+02]
 [ 3.74497361e+01  1.56955019e+00  3.03106970e+02]
 [ 1.82524003e-01  1.90822509e+00  2.29410001e+02]
 [ 5.63869846e-02  1.04253233e+00  2.52369688e+02]
 [ 7.81793242e-01  2.43874406e+00  2.32534357e+02]
 [-3.17327268e-01  3.29272760e+00  2.48979807e+02]
 [ 6.07604079e-01  2.81357238e+00  2.28249595e+02]
 [ 5.75916562e-01  4.25450508e+00  1.98928053e+02]
 [ 4.45539535e-01  4.61406463e+00  1.92544520e+02]
 [-1.55274812e-01  3.49668651e+00  2.54454246e+02]
 [ 9.28060668e-02  1.55996035e+00  2.32340778e+02]
 [-3.53715764e-01  3.17814162e+00  2.49617939e+02]
 [ 3.91591250e-01  1.81062953e+01  2.27930639e+02]
 [-7.91780248e-01  1.60348954e+00  2.17748088e+02]
 [ 7.65656713e+00  2.68952268e+00  2.96147697e+02]
 [-2.32387605e-01  8.59818457e-01  2.55478774e+02]
 [ 1.78005146e-01 

In [6]:
print(pd.DataFrame(y_pred, columns=["heatwave_days_pred", "precip_days_heavy_pred", "dry_days_pred"]))
print("\nModel coefficients:", model.named_steps["regressor"].coef_.shape)

    heatwave_days_pred  precip_days_heavy_pred  dry_days_pred
0             0.092799                2.254943     213.433457
1             0.874100                3.524735     262.935932
2             0.705318                3.395572     260.750375
3            -0.073587                2.700828     233.908964
4            37.449736                1.569550     303.106970
..                 ...                     ...            ...
71           -0.291399                2.269323     219.891270
72           -0.247632                3.142254     257.915779
73            0.602947                4.908392     288.601341
74            0.361972                1.963832     213.336187
75            0.190468                3.433883     235.195754

[76 rows x 3 columns]

Model coefficients: (3, 32)


In [7]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# R² for each target
print("R² scores:")
print(r2_score(y_test, y_pred, multioutput='raw_values'))

# RMSE for each target
print("\nRMSE:")
print(np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values')))

# MAE for each target
print("\nMAE:")
print(mean_absolute_error(y_test, y_pred, multioutput='raw_values'))

R² scores:
[0.84199152 0.66277089 0.64596172]

RMSE:
[ 3.05410116  2.2854572  18.26800491]

MAE:
[ 1.31510347  1.80420544 15.51691538]


In [8]:
comparison = pd.DataFrame({
    "heatwave_actual": y_test["heatwave_days"].values,
    "heatwave_pred": y_pred[:, 0],
    "precip_actual": y_test["precip_days_heavy"].values,
    "precip_pred": y_pred[:, 1],
    "dry_actual": y_test["dry_days"].values,
    "dry_pred": y_pred[:, 2],
})

print(comparison.head(10))

   heatwave_actual  heatwave_pred  precip_actual  precip_pred  dry_actual  \
0                0       0.092799              5     2.254943         191   
1                0       0.874100              6     3.524735         272   
2                0       0.705318              3     3.395572         289   
3                0      -0.073587              1     2.700828         256   
4               29      37.449736              1     1.569550         303   
5                0       0.182524              1     1.908225         211   
6                0       0.056387              0     1.042532         246   
7                1       0.781793              1     2.438744         265   
8                0      -0.317327              1     3.292728         275   
9                0       0.607604              2     2.813572         235   

     dry_pred  
0  213.433457  
1  262.935932  
2  260.750375  
3  233.908964  
4  303.106970  
5  229.410001  
6  252.369688  
7  232.534357  
8  248.9